<a href="https://colab.research.google.com/github/Trangnguyen1402/AAI2025/blob/2026Fall/customer_churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# Data source:
# IBM Telco Customer Churn Dataset
# File: WA_Fn-UseC_-Telco-Customer-Churn.csv

# Load the dataset
df = pd.read_csv(
    "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

# Convert TotalCharges to numbers
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Convert churn labels into numbers
# Yes = 1, No = 0
df["Churn"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

# Remove missing values
df = df.dropna(
    subset=[
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "SeniorCitizen",
        "Contract",
        "InternetService",
        "PaymentMethod",
        "Churn"
    ]
)

# Use 200 records while keeping both churn classes
df, unused_data = train_test_split(
    df,
    train_size=200,
    stratify=df["Churn"],
    random_state=42
)

print(f"Number of records used: {len(df)}")

# Numerical features
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "SeniorCitizen"
]

# Categorical features
categorical_features = [
    "Contract",
    "InternetService",
    "PaymentMethod"
]

# Features and target
X = df[
    numeric_features + categorical_features
]

y = df["Churn"]

# Scale numerical features and encode categorical features
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

# Create logistic regression pipeline
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                random_state=42,
                max_iter=1000
            )
        )
    ]
)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Train the model
model.fit(X_train, y_train)

# Test the model
test_predictions = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    test_predictions
)

print(f"Model Accuracy: {accuracy:.2f}")

# Create a new customer
new_customer = pd.DataFrame({
    "tenure": [12],
    "MonthlyCharges": [75.00],
    "TotalCharges": [900.00],
    "SeniorCitizen": [0],
    "Contract": ["Month-to-month"],
    "InternetService": ["Fiber optic"],
    "PaymentMethod": ["Electronic check"]
})

# Predict churn probability
churn_probability = model.predict_proba(
    new_customer
)[0][1]

# Use a 0.5 threshold
threshold = 0.5

if churn_probability >= threshold:
    churn_prediction = 1
else:
    churn_prediction = 0

print(
    f"\nChurn probability for new customer: "
    f"{churn_probability:.2f}"
)

print(
    f"Churn prediction "
    f"(1 = churn, 0 = no churn): "
    f"{churn_prediction}"
)

# Business interpretation
if churn_prediction == 1:
    print(
        "Business interpretation: "
        "This customer is at risk of leaving. "
        "The company could offer a discount, upgrade, "
        "or personalized retention offer."
    )
else:
    print(
        "Business interpretation: "
        "This customer is not currently classified as "
        "high risk for churn."
    )

# Display model coefficients
feature_names = (
    model.named_steps["preprocessor"]
    .get_feature_names_out()
)

coefficients = (
    model.named_steps["classifier"]
    .coef_[0]
)

print("\nModel Coefficients:")

for feature, coefficient in zip(
    feature_names,
    coefficients
):
    print(f"{feature}: {coefficient:.2f}")

Number of records used: 200
Model Accuracy: 0.75

Churn probability for new customer: 0.61
Churn prediction (1 = churn, 0 = no churn): 1
Business interpretation: This customer is at risk of leaving. The company could offer a discount, upgrade, or personalized retention offer.

Model Coefficients:
numeric__tenure: -1.20
numeric__MonthlyCharges: 0.40
numeric__TotalCharges: 0.22
numeric__SeniorCitizen: 0.06
categorical__Contract_Month-to-month: 0.64
categorical__Contract_One year: -0.23
categorical__Contract_Two year: -0.41
categorical__InternetService_DSL: -0.75
categorical__InternetService_Fiber optic: 0.71
categorical__InternetService_No: 0.04
categorical__PaymentMethod_Bank transfer (automatic): 0.19
categorical__PaymentMethod_Credit card (automatic): -0.16
categorical__PaymentMethod_Electronic check: 0.14
categorical__PaymentMethod_Mailed check: -0.18
